# 03 — In-Vehicle Phone-to-Vehicle Alignment & Leveling

**SIH PS 26168 — Intelligent Dead Reckoning**

> **Section 21:** In-vehicle alignment & calibration engine.
> **Section 10:** Notebook-first traceable evaluation.

### Objectives:
1. Estimate roll and pitch leveling angles from gravity vector during stationary state.
2. Estimate yaw misalignment relative to vehicle longitudinal axis from forward acceleration events.
3. Transform phone body acceleration into vehicle forward, lateral, and vertical axes.
4. Validate alignment against vehicle CAN-bus longitudinal acceleration reference.

## 1. Imports & Configuration

In [1]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.data_loader import IOVNBDLoader
from src.calibration.alignment import PhoneVehicleAlignment

plots_dir = PROJECT_ROOT / 'plots' / 'alignment'
plots_dir.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', font_scale=1.1)
print('Alignment modules loaded.')

Alignment modules loaded.


## 2. Load Session & Run Dynamic Calibration

In [2]:
loader = IOVNBDLoader()
session_id = 'M'
sess = loader.load_session(session_id, preprocess_imu=True)

aligner = PhoneVehicleAlignment()
R_p_to_v = aligner.calibrate(
    sess['accel_raw'],
    zupt_mask=sess['zupt_mask'],
    velocity_ref=sess['vehicle']['speed_mps']
)

print('=' * 60)
print(f'ALIGNMENT RESULTS — SESSION {session_id}')
print('=' * 60)
print(f'Roll Angle (Leveling)  : {aligner.alignment_angles_deg["roll"]:+.2f}°')
print(f'Pitch Angle (Leveling) : {aligner.alignment_angles_deg["pitch"]:+.2f}°')
print(f'Yaw Angle (Heading)    : {aligner.alignment_angles_deg["yaw"]:+.2f}°')
print(f'Rotation Matrix R_p_to_v:\n{R_p_to_v.round(4)}')
print('=' * 60)

ALIGNMENT RESULTS — SESSION M
Roll Angle (Leveling)  : +0.24°
Pitch Angle (Leveling) : -0.48°
Yaw Angle (Heading)    : -40.61°
Rotation Matrix R_p_to_v:
[[ 0.7591  0.6509 -0.0091]
 [-0.6509  0.7591  0.0022]
 [ 0.0083  0.0042  1.    ]]


## 3. Transform Acceleration to Vehicle Frame

In [3]:
# Transform linear acceleration into vehicle frame
# Vehicle Frame: X = Forward, Y = Lateral, Z = Vertical
accel_veh = aligner.transform_to_vehicle(sess['accel_linear'])

# Select a driving window with acceleration & deceleration events
w_start = 2000
w_end   = 2500
t_slice = sess['time_s'][w_start:w_end] - sess['time_s'][w_start]

fig, axs = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

# Forward Acceleration vs CAN-bus reference
axs[0].plot(t_slice, accel_veh[w_start:w_end, 0], color='#1f77b4', linewidth=1.5, label='Aligned Phone Forward Accel')
if sess['vehicle']['lon_accel_mps2'] is not None:
    axs[0].plot(t_slice, sess['vehicle']['lon_accel_mps2'][w_start:w_end], color='red', linestyle='--', alpha=0.8, label='CAN-bus Lon Accel Ref')
axs[0].set_ylabel('Forward Accel\n[m/s²]')
axs[0].legend(loc='upper right')
axs[0].set_title('Vehicle Frame Acceleration Comparison')

# Lateral Acceleration
axs[1].plot(t_slice, accel_veh[w_start:w_end, 1], color='#ff7f0e', linewidth=1.5, label='Aligned Phone Lateral Accel')
if sess['vehicle']['lat_accel_mps2'] is not None:
    axs[1].plot(t_slice, sess['vehicle']['lat_accel_mps2'][w_start:w_end], color='darkred', linestyle='--', alpha=0.8, label='CAN-bus Lat Accel Ref')
axs[1].set_ylabel('Lateral Accel\n[m/s²]')
axs[1].legend(loc='upper right')

# Vehicle Speed
if sess['vehicle']['speed_mps'] is not None:
    axs[2].plot(t_slice, sess['vehicle']['speed_mps'][w_start:w_end] * 3.6, color='#2ca02c', linewidth=1.8, label='Vehicle Speed [km/h]')
axs[2].set_ylabel('Speed [km/h]')
axs[2].set_xlabel('Elapsed Time [seconds]')
axs[2].legend(loc='upper right')

plt.tight_layout()
plt.savefig(plots_dir / f'alignment_validation_{session_id}.png', dpi=200)
plt.close()
print(f'Alignment validation plot saved to: plots/alignment/alignment_validation_{session_id}.png')

Alignment validation plot saved to: plots/alignment/alignment_validation_M.png


## 4. Multi-Driver Calibration Stability Evaluation

In [4]:
# Run alignment on multiple distinct sessions
test_stems = ['M', 'S1', 'Vfa01', 'Vta1a']
results = []

for stem in test_stems:
    if stem not in loader.sessions: continue
    s = loader.load_session(stem, preprocess_imu=True)
    alg = PhoneVehicleAlignment()
    alg.calibrate(s['accel_raw'], zupt_mask=s['zupt_mask'], velocity_ref=s['vehicle']['speed_mps'])
    results.append({
        'session': stem,
        'driver': s['driver'],
        'roll_deg': alg.alignment_angles_deg['roll'],
        'pitch_deg': alg.alignment_angles_deg['pitch'],
        'yaw_deg': alg.alignment_angles_deg['yaw'],
    })

df_res = pd.DataFrame(results)
print('\n' + '=' * 60)
print('MULTI-SESSION ALIGNMENT COMPARISON')
print('=' * 60)
print(df_res.to_string(index=False))
print('=' * 60)
print('Alignment notebook completed successfully.')


MULTI-SESSION ALIGNMENT COMPARISON
session       driver  roll_deg  pitch_deg  yaw_deg
      M M (Driver B)      0.24      -0.48   -40.61
     S1           S1     -0.22       0.29   -44.62
  Vfa01      V-Vfa01      1.91       0.01   108.80
  Vta1a       Vta01a      1.42       0.64    80.72
Alignment notebook completed successfully.
